# Análise de editais
Lê os arquivos de um edital (PDF, DOCX, ODT) e gera um checklist do que é exigido.

**Antes de rodar:** guarde a chave do Google AI Studio em 🔑 *Secrets* com o nome `GEMINI_API_KEY` e ative *Notebook access*.

## 1. Instalação e código do projeto

In [ ]:
!pip install -q pydantic pymupdf python-docx odfpy google-genai

# Envie a pasta 'src' (zip) do projeto. Ex.: zip -r src.zip src
import os
if not os.path.isdir('src'):
    from google.colab import files
    up = files.upload()  # selecione src.zip
    !unzip -q -o src.zip

## 2. Envio dos arquivos do edital
Envie o edital e os anexos do mesmo processo (PDF, DOCX ou ODT).

In [ ]:
from pathlib import Path
from google.colab import files

Path('editais').mkdir(exist_ok=True)
enviados = files.upload()
caminhos = []
for nome, conteudo in enviados.items():
    destino = Path('editais') / nome
    destino.write_bytes(conteudo)
    caminhos.append(destino)
caminhos

## 3. Extração do checklist

In [ ]:
from google.colab import userdata
from src.extracao import extrair_com_gemini
from src.processo import processar

API_KEY = userdata.get('GEMINI_API_KEY')

def extrator(texto, arquivo, caminho, escaneado):
    return extrair_com_gemini(texto, arquivo, caminho=caminho, escaneado=escaneado, api_key=API_KEY)

checklist = processar(caminhos, extrator)
len(checklist.exigencias)

## 4. Resultado

In [ ]:
from IPython.display import Markdown, display
from src.relatorio_edital import para_markdown, salvar

display(Markdown(para_markdown(checklist)))
salvar(checklist, 'saida')  # gera saida/checklist.json e saida/checklist.md

In [ ]:
# Opcional: baixar os resultados
!zip -q -r saida.zip saida
files.download('saida.zip')